This notebook aim to help getting started with ``xLLiM`` library. It presents a complete and detailed workflow with a simple example. The workflow can be summed up into 3 main section:
 1. Generate a training dataset (optional)
 2. Train your GLLiM model
 3. Prediction
 4. Apply importance sampling (optional)
 5. Post treatment analysis

#### Import xLLiM module

In [1]:
import xllim
import numpy as np
import matplotlib.pyplot as plt
import logging
logging.getLogger().setLevel(logging.INFO)

## 1. Generate a training Dataset

=> If you have a ready-to-use Dataset, go to next section. 

Otherwise, you can generate a training dataset with xllim library from physical models. For now there a two concrete Photometric models implemented (Hapke and Shkuratov), one very basic Test model and a custom class (ExternalPythonModel) where you can define your own functional model in Python language.
More details are available in [documentation](https://xllim-tools.github.io/xllim/functional/functional_model.html)


#### Create a functional model

In [2]:
model = xllim.TestModel()

You can use your model to apply basic class methods :
 - **F** : the functional model F describing the functional model.
 - **getDimensionY** : returns the dimension of Y
 - **getDimensionX** : return de dimension of X
 - **toPhysic** : converts the X data from mathematical framework (0<X<1) to physical framework
 - **fromPhysic** : converts the X data from physical framework to mathematical framework (0<X<1)

In [ ]:
L = model.getDimensionX()
D = model.getDimensionY()
print("Test model transform a vector X of dimension L = {} into a vector Y of dimension D = {}".format(L, D))

x = np.ones(L)
x_physic = model.toPhysic(x)
x_mat = model.fromPhysic(x_physic)
print("{} = model.toPhysic({})".format(x_physic, x))
print("{} = model.fromPhysic({})".format(x_mat, x_physic))

x = np.zeros(L)
e = np.exp(1)
exact_y = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
y = model.F(x)
print("model.F({}) = ".format(x))
print(y)
print("The exact result of TestModel([0,0,0,0]) is ")
print(exact_y)


#### Generate a dataset from the functional model

The dataset is composed of 
 - x_gen with shape (L, N)
 - y_gen with shape (D, N)


In [ ]:
N = 10_000                                          # number of generated observation
generator_type = "sobol"                            # the type of the generator used to generate x_gen matrix values
covariance = np.ones(model.getDimensionY()) * 1e-5  # vector of dimension D coresponding to the y_i variances.
seed = 12345                                        # seed number for random generators

x_gen, y_gen = model.genData(N, generator_type, covariance, seed)

print(x_gen.shape)
print(y_gen.shape)

## 2. Train your GLLiM model

##### Instanciate a GLLiM object
You can instanciate a GLLiM object according with constraints.
Refer to the documentation for more details on the notation and the mathematical aspect

In [ ]:
# Choose Covariance matrix type
gamma_type = "Full"
sigma_type = "Diag"

# Define model dimensions
# ! D and L have to fit the dataset dimensions
K = 5              # number of gaussians in the mixture model
D = y_gen.shape[0]  # = model.getDimensionY()
L = x_gen.shape[0]  # = model.getDimensionX()

# Hybrid model
n_hidden_variables = 0
L_t = L - n_hidden_variables
L_w = n_hidden_variables


gllim = xllim.GLLiM(
    K, D, L, gamma_type="full", sigma_type="diag", n_hidden_variables=n_hidden_variables
)


You can set an intial theta to your model before the initialisation and/or the training step if you want. Theta is initialized with default settings as, $\forall$ k $\in$ [0,K-1] :
 - Pi[k] = 1/K
 - A[k] = $O_{D,L}$
 - B[k] = $O_{D}$
 - C[k] = $O_{L}$
 - Gamma[k] = $I_{L,L}$
 - Sigma[k] = $I_{D,D}$

In [ ]:
# print(gllim.getParams())
# print(gllim.getParamPi())
# print(gllim.getParamA())
print(gllim.getParamC())
# print(gllim.getParamGamma())
# print(gllim.getParamB())
# print(gllim.getParamSigma())

gllim.setParamC(np.ones((K, L)) * 0.02)

print(gllim.getParamC())

Apply GLLiM initialisation step

In [ ]:
# initialisation parameters
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 5
gmm_em_iteration = 10
gmm_floor = 1e-12
nb_experiences = 5

seed = 12345678
verbose = 1

gllim.initialize(
    x_gen,
    y_gen,
    gllim_em_iteration,
    gllim_em_floor,
    gmm_kmeans_iteration,
    gmm_em_iteration,
    gmm_floor,
    nb_experiences,
    seed,
    verbose,
)

Train GLLiM model with GLLiM-EM algorithm

In [ ]:
# training parameters
train_max_iteration = 100
train_ratio_ll = 1e-5
train_floor = 1e-12

gllim.train(
    x_gen, y_gen, train_max_iteration, train_ratio_ll, train_floor, verbose
)


Once trained you can retrieve the inverted model parameters $\theta^*$

In [ ]:
theta_star = gllim.getInverse()
print(theta_star.Pi)

## 3. Prediction

You can use your trained GLLiM in order to evaluate probability densites :
 - directDensities(x) correspond to the **forward conditional density** and returns $E[Y = y|X = x; \theta]$
 - inverseDensities(x) correspond to the **backward** or **inverse conditional density** and returns $E[Y = y|X = x; \theta]$

In the case your dataset is represented by a *functional model*, GLLiM model is your surrogate model analytically inversible. You can thus evaluate very quickly the inverted F model on a large number of observed Y :

$x = F^{-1}(y) \approx GLLiM^{-1}(y)$

#### Let's generate a test dataset for example purpose

In the example below we generate 100 observations from the *Test model* through the **genData** method (see section "DataGeneration"). In concrete life case, the obsered $Y$ can come with associated incertitude. That is why a random incertitude vector is associated to all $y$ vectors. This is optional.

In [ ]:
N_obs = 100
N_obs_arange = np.arange(N_obs)
# x_obs, y_obs = model.genData(N_obs, "sobol", np.zeros(model.getDimensionY()), 12345)
y_incertitudes = np.random.rand(D, N_obs) * 0.01

# X parameters
x_obs = np.empty((L, N_obs))
x_obs[0] = 0.58
x_obs[1] = 0.4 * np.sin(2. * np.pi * N_obs_arange / N_obs) + 0.5
x_obs[2] = 0.4 * np.sin(2. * np.pi * N_obs_arange / N_obs + np.pi) + 0.5
x_obs[3] = 0.2 * np.cos(2. * np.pi * N_obs_arange / N_obs) + 0.3

# Y parameters
y_obs = np.empty((D, N_obs))
y_obs_noise = np.empty((D, N_obs))
y_obs_noised = np.empty((D, N_obs))
for n in range(N_obs):
    y_obs[:,n] = model.F(x_obs[:,n])
    # Add noise for each Y component
    for d in range(D):
        y_obs_noise[d,n] = (y_obs[d,n]/1000.) * np.random.normal(0, np.power(y_obs[d,n]/1000., 2), 1)
y_obs_noised = y_obs + y_obs_noise


### Plot results ###
fig, axs = plt.subplots(1, L, figsize=(15, 5))
fig.suptitle("X Parameters", fontsize=16)

for l in range(L):
    axs[l].plot(N_obs_arange, x_obs[l,:], 'r-')
    axs[l].set_xlabel("Observations")
    axs[l].set_title('X_'+ str(l))
    
plt.show()


fig, axs = plt.subplots(1, D, figsize=(30, 5))
fig.suptitle("Y = F(X)", fontsize=16)

for d in range(D):
    axs[d].plot(N_obs_arange, y_obs_noised[d,:], 'b-')
    axs[d].set_xlabel("Observations")
    axs[d].set_title('Y_'+ str(d))
    
plt.show()


Then we apprly the inversion

In [ ]:
predictions = gllim.inverseDensities(y_obs_noised, y_obs_noise)

# comparing to x_obs
print("x_obs[0] = {}".format(x_obs[:,0]))
print("x_predicted[0] = {}".format(predictions.fullGMM.mean[:,0]))

In [ ]:
type(predictions.mergedGMM.covs)

## 4. Importance sampling

The sampling step depends on a *functional model*. If our case we try to improve our result thank to the IMIS (Incremental Mixture Importance Sampling) algorithm 

In [ ]:
# Covariance on the target density
covariance = np.ones(model.getDimensionY()) * 0.001

# Importance sampling parameters
N_0 = 1000
B = 500
J = 20

is_results = model.importanceSampling(predictions.fullGMM, y_obs_noised, y_obs_noise, N_0, B, J, covariance)

In [ ]:
# comparing to x_obs
print("x_obs[0] = {}".format(x_obs[:,0]))
print("x_predicted[0] = {}".format(predictions.fullGMM.mean[:,0]))
print("x_is[0] = {}".format(is_results.predictions[:,0]))

## 5. Post-processing analysis

#### Evaluate reconstruction error and error on parameters

In [ ]:
def compute_reconstruction_error(reconstruction, observation):
    return np.linalg.norm(observation - reconstruction) / np.linalg.norm(observation)

reconstruction_error = []
reconstruction_error_is = []
for n in range(N_obs):
    reconstruction_error.append(
        compute_reconstruction_error(model.F(predictions.fullGMM.mean[:,n]), y_obs_noised[:,n])
    )
    reconstruction_error_is.append(
        compute_reconstruction_error(model.F(is_results.predictions[:,n]), y_obs_noised[:,n])
    )

error_on_param = []
error_on_param_is = []
for l in range(L):
    temp = 0
    temp_is = 0
    for n in range(N_obs):
        temp += compute_reconstruction_error(predictions.fullGMM.mean[l,n], x_obs[l,n])
        temp_is += compute_reconstruction_error(is_results.predictions[l,n], x_obs[l,n])
    error_on_param.append(temp / N_obs)
    error_on_param_is.append(temp_is / N_obs)
    

fig, axs = plt.subplots(1, 2, figsize=(20, 5))
fig.suptitle("Reconstruction error", fontsize=16)

axs[0].plot(N_obs_arange, reconstruction_error, 'b*', label='GLLiM prediction')
axs[0].plot(N_obs_arange, reconstruction_error_is, 'g+', label='GLLiM + IS')
axs[0].set_xlabel("Observations")
axs[0].set_title("Reconstruction error for each observation")
axs[0].legend()

axs[1].plot(np.arange(L), error_on_param, 'b*', label='GLLiM prediction')
axs[1].plot(np.arange(L), error_on_param_is, 'g+', label='GLLiM + IS')
axs[1].set_xlabel("Observations")
axs[1].set_title("Error for each parameter")
axs[1].legend()
    
plt.show()

#### Comparing estimation methods (GLLiM prediction and GLLiM + IS prediction)

In [ ]:
### Plot results ###
fig, axs = plt.subplots(1, L, figsize=(15, 5))
fig.suptitle("X Parameters estimations", fontsize=16)

for l in range(L):
    axs[l].plot(N_obs_arange, x_obs[l,:], 'k-', label='Original')
    axs[l].plot(N_obs_arange, predictions.fullGMM.mean[l], 'b-', label='GLLiM prediction')
    axs[l].plot(N_obs_arange, is_results.predictions[l], 'g-', label='GLLiM + IS')
    axs[l].set_xlabel("Observations")
    axs[l].set_title('X_'+ str(l))
    axs[l].legend()
    
plt.show()